# Full-res CACHE builder + uploader (CPU session — no GPU quota)

Builds the training cache ONCE and uploads it as private Kaggle datasets:
X/y fp16 zarr chunks + `X_test.nc` + a 1961-1990 hist window (all the indices ever read).

**Setup:**
1. Right panel → **Data → Add Input → Datasets** → `climx-full-resolution-data` → Add
2. **Accelerator: NONE (CPU)** — this job never touches the GPU quota

**RESUMABLE**: re-run until `CACHE_UPLOAD_OK`; uploaded files are deleted from /tmp and
skipped on re-runs. Ends by printing the dataset slugs + manifest — commit
`data/full_cache_manifest.json` to the repo (or send it to None).

**Runtime**: ~4-6 h (X cache ~1.5 h + slab y reads + hist window + uploads).

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run(["find", repo, "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— MUST be the commit None told you")
print("setup done")

## 2. Build + upload the cache

In [ ]:
%run scripts/full_cache.py --subsample 3

## 3. After CACHE_UPLOAD_OK

The log prints the dataset slugs (`nonewanitchollakit/climx-cache-b1`, …) and the
manifest JSON. Commit the manifest to the repo and share the datasets with the team
(dataset page → Share → add collaborators) so training sessions can attach them.